# MLP com Backpropagation — Implementação e Comparação com Scikit-Learn

Neste notebook implementamos um **Multi-Layer Perceptron (MLP)** do zero com o algoritmo de
backpropagation, comparando seus resultados com o `MLPClassifier` do Scikit-Learn.

O MLP customizado aceita parâmetros para definir:
- a **quantidade de camadas ocultas** e o número de **neurônios em cada camada** (`n_hidden`)
- a **função de ativação de cada camada** individualmente (`activations`)

In [22]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelBinarizer
from sklearn.neural_network import MLPClassifier
import numpy as np
from scipy.special import expit
from abc import ABC, abstractmethod

## 1. Dataset

Usamos o **Digits** do Scikit-Learn: imagens 8×8 de dígitos manuscritos (0–9),
totalizando 1.797 amostras com 64 atributos cada.

In [23]:
X, y = load_digits(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Normaliza para [0, 1]
X_train = X_train / 16.0
X_test  = X_test  / 16.0
print(f"Treino: {X_train.shape}, Teste: {X_test.shape}")

Treino: (1437, 64), Teste: (360, 64)


## 2. Implementação do MLP

### 2.1 Utilitários

In [24]:
def adicionar_bias(X):
    return np.c_[np.ones(X.shape[0]), X]

def sign(a):
    return (a >= 0) * 2 - 1

### 2.2 Funções de Ativação

Implementamos uma interface base abstrata e duas funções de ativação concretas:
**Tanh** e **Sigmoid**. Qualquer nova função pode ser adicionada seguindo o mesmo contrato.

In [25]:
class FuncaoAtivacao(ABC):
    @abstractmethod
    def forward(self, a):
        pass

    @abstractmethod
    def backward(self, a):
        pass

class Tanh(FuncaoAtivacao):
    """Tangente hiperbólica no intervalo (-1, 1)."""
    def forward(self, a):
        return expit(a) * 2 - 1

    def backward(self, a):
        return 1 - np.square(self.forward(a))

class Sigmoid(FuncaoAtivacao):
    """Função logística no intervalo (0, 1)."""
    def forward(self, a):
        return expit(a)

    def backward(self, a):
        s = self.forward(a)
        return s * (1 - s)

### 2.3 Algoritmo de Backpropagation

A classe `Backprop` executa o forward pass, o backward pass (cálculo dos gradientes)
e a atualização dos pesos por gradiente descendente com regularização L2.

**Parâmetros principais:**
| Parâmetro | Descrição |
|---|---|
| `n_hidden` | Lista com o nº de neurônios em cada camada oculta. Ex: `[64, 32]` = duas camadas |
| `activations` | Lista de objetos `FuncaoAtivacao`, um para cada camada (incluindo saída) |
| `learning_rate` | Taxa de aprendizado |
| `max_iter` | Número de épocas de treinamento |
| `regularization` | Coeficiente de regularização L2 (penaliza pesos grandes) |

In [26]:
class Backprop:
    def __init__(
        self,
        max_iter=1000,
        learning_rate=0.01,
        n_hidden=[64],
        regularization=0,
        activations=None
    ):
        self.max_iter      = max_iter
        self.learning_rate = learning_rate
        self.n_hidden      = n_hidden
        self.regularization = regularization

        num_layers = len(n_hidden) + 1
        self.activations = activations if activations is not None             else [Tanh() for _ in range(num_layers)]

    # ── Forward pass
    def forward(self, X):
        self.A, self.Z = [], []
        saida = X.copy()
        for i, W in enumerate(self.Ws):
            self.A.append(adicionar_bias(saida))
            self.Z.append(self.A[-1] @ W)
            saida = self.activations[i].forward(self.Z[-1])
        return saida

    # ── Backward pass (cálculo dos gradientes)
    def backward(self, X, y, y_pred):
        grads = []
        delta = (y - y_pred) * self.activations[-1].backward(self.Z[-1])
        grads.insert(0, self.A[-1].T @ delta)

        for i in range(len(self.Ws) - 1, 0, -1):
            delta = (delta @ self.Ws[i][1:, :].T) * self.activations[i - 1].backward(self.Z[i - 1])
            grads.insert(0, self.A[i - 1].T @ delta)

        # Atualização dos pesos com regularização L2
        for i in range(len(self.Ws) - 1, -1, -1):
            self.Ws[i] *= 1 - self.regularization * self.learning_rate
            self.Ws[i] += grads[i] * self.learning_rate

    # ── Inicialização dos pesos e loop de treinamento
    def fit(self, X, y):
        self.Ws = []
        entrada = X.shape[1]

        for tamanho_camada in self.n_hidden:
            scale = np.sqrt(2.0 / (entrada + tamanho_camada))
            self.Ws.append(np.random.randn(entrada + 1, tamanho_camada) * scale)
            entrada = tamanho_camada

        if y.ndim == 1:
            y = y.reshape(-1, 1)
        scale_out = np.sqrt(2.0 / (entrada + y.shape[1]))
        self.Ws.append(np.random.randn(entrada + 1, y.shape[1]) * scale_out)

        for _ in range(self.max_iter):
            y_pred = self.forward(X)
            self.backward(X, y, y_pred)

### 2.4 Classe MLP (wrapper compatível com Scikit-Learn)

`CustomMLP` encapsula o `Backprop` e expõe a interface padrão `fit` / `predict`,
compatível com o ecossistema do Scikit-Learn (herda `BaseEstimator` e `ClassifierMixin`).

In [27]:
class CustomMLP(BaseEstimator, ClassifierMixin):
    def __init__(self, algoritmo=None):
        self.algoritmo = algoritmo if algoritmo is not None else Backprop()

    def fit(self, X, y):
        self.binarizador = LabelBinarizer()
        y_bin = self.binarizador.fit_transform(y)
        self.algoritmo.fit(X, y_bin)
        return self

    def predict(self, X):
        y_pred = self.algoritmo.forward(X)

        if y_pred.shape[1] == 1:
            y_pred = sign(y_pred)
        else:
            mascara = np.zeros(y_pred.shape)
            indices = np.argmax(y_pred, axis=1)
            mascara[np.arange(y_pred.shape[0]), indices] = 1
            y_pred = mascara

        return self.binarizador.inverse_transform(y_pred)

## 3. Testes do MLP Customizado

### 3.1 Sem camada oculta (Perceptron simples)
`n_hidden=[]` produz um modelo linear (sem camada oculta).

In [28]:
np.random.seed(42)
modelo1 = CustomMLP(
    algoritmo=Backprop(
        max_iter=1000,
        learning_rate=0.001,
        n_hidden=[], # sem camadas ocultas
        regularization=0.01,
        activations=[Sigmoid()]
    )
)
modelo1.fit(X_train, y_train)

print("=== Sem camada oculta (Tanh) ===")
print(f"Acurácia Treino: {accuracy_score(y_train, modelo1.predict(X_train)):.4f}")
print(f"Acurácia Teste:  {accuracy_score(y_test,  modelo1.predict(X_test)):.4f}")

=== Sem camada oculta (Tanh) ===
Acurácia Treino: 0.9638
Acurácia Teste:  0.9528


### 3.2 Uma camada oculta — Tanh + Sigmoid
`n_hidden=[64]` → uma camada oculta com 64 neurônios.

In [29]:
np.random.seed(42)
modelo2 = CustomMLP(
    algoritmo=Backprop(
        max_iter=1500,
        learning_rate=0.005,
        n_hidden=[64], # 1 camada oculta, 64 neurônios
        regularization=0.01,
        activations=[Tanh(), Sigmoid()] # ativação camada oculta, ativação saída
    )
)
modelo2.fit(X_train, y_train)

print("=== 1 camada oculta: 64 neurônios | Tanh -> Sigmoid ===")
print(f"Acurácia Treino: {accuracy_score(y_train, modelo2.predict(X_train)):.4f}")
print(f"Acurácia Teste:  {accuracy_score(y_test,  modelo2.predict(X_test)):.4f}")

=== 1 camada oculta: 64 neurônios | Tanh -> Sigmoid ===
Acurácia Treino: 0.1447
Acurácia Teste:  0.1833


### 3.3 Duas camadas ocultas — Sigmoid + Tanh + Sigmoid
`n_hidden=[128, 64]` → duas camadas ocultas (128 e 64 neurônios, respectivamente).

In [31]:
np.random.seed(42)
modelo3 = CustomMLP(
    algoritmo=Backprop(
        max_iter=2000,
        learning_rate=0.003,
        n_hidden=[128, 64], # 2 camadas ocultas
        regularization=0.01,
        activations=[Sigmoid(), Tanh(), Sigmoid()] # oculta1, oculta2, saída
    )
)
modelo3.fit(X_train, y_train)

print("=== 2 camadas ocultas: 128 → 64 | Sigmoid -> Tanh -> Sigmoid ===")
print(f"Acurácia Treino: {accuracy_score(y_train, modelo3.predict(X_train)):.4f}")
print(f"Acurácia Teste:  {accuracy_score(y_test,  modelo3.predict(X_test)):.4f}")

=== 2 camadas ocultas: 128 → 64 | Sigmoid -> Tanh -> Sigmoid ===
Acurácia Treino: 0.7850
Acurácia Teste:  0.7917


## 4. Comparação com o `MLPClassifier` do Scikit-Learn

Para uma comparação justa, testamos configurações equivalentes às do nosso MLP.

In [32]:
print("=== Scikit-Learn MLPClassifier ===\n")

# Sem camadas ocultas
sk1 = MLPClassifier(hidden_layer_sizes=(), max_iter=1000, random_state=42)
sk1.fit(X_train, y_train)
print("Sem camada oculta:")
print(f"  Treino: {accuracy_score(y_train, sk1.predict(X_train)):.4f}")
print(f"  Teste:  {accuracy_score(y_test,  sk1.predict(X_test)):.4f}")

# 1 camada oculta
sk2 = MLPClassifier(hidden_layer_sizes=(64,), activation='tanh', max_iter=1500, random_state=42)
sk2.fit(X_train, y_train)
print("\n1 camada oculta (64) — Tanh:")
print(f"  Treino: {accuracy_score(y_train, sk2.predict(X_train)):.4f}")
print(f"  Teste:  {accuracy_score(y_test,  sk2.predict(X_test)):.4f}")

# 2 camadas ocultas
sk3 = MLPClassifier(hidden_layer_sizes=(128, 64), activation='logistic', max_iter=2000, random_state=42)
sk3.fit(X_train, y_train)
print("\n2 camadas ocultas (128, 64) — Sigmoid:")
print(f"  Treino: {accuracy_score(y_train, sk3.predict(X_train)):.4f}")
print(f"  Teste:  {accuracy_score(y_test,  sk3.predict(X_test)):.4f}")

=== Scikit-Learn MLPClassifier ===



/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


Sem camada oculta:
  Treino: 0.9930
  Teste:  0.9694

1 camada oculta (64) — Tanh:
  Treino: 1.0000
  Teste:  0.9694

2 camadas ocultas (128, 64) — Sigmoid:
  Treino: 1.0000
  Teste:  0.9750


## 5. Resumo dos Resultados

| Modelo | Camadas ocultas | Ativação | Acurácia Treino | Acurácia Teste |
|---|---|---|---|---|
| CustomMLP | Nenhuma | Sigmoid | 0.9638 | 0.9528 |
| CustomMLP | 64 | Tanh → Sigmoid | 0.1447 | 0.1833 |
| CustomMLP | 128 → 64 | Sigmoid → Tanh → Sigmoid | 0.7850 | 0.7917 |
| Sklearn MLP | Nenhuma | — | 0.9930 | 0.9694 |
| Sklearn MLP | 64 | Tanh | 1.0000 | 0.9694 |
| Sklearn MLP | 128 → 64 | Sigmoid | 1.0000 | 0.9750 |


**Conclusão:** O MLP implementado do zero segue a mesma lógica de backpropagation,
porém o Scikit-Learn tende a alcançar resultados melhores graças a otimizações
como mini-batches, inicialização de pesos inteligente (Glorot) e solver Adam.